# Laboratorio 12b — SRGAN desde cero (solo programación)
**Redes Neuronales y Aprendizaje Profundo — UNI FIIS, Unidad de Posgrado**
Aldo Camargo Fernández-Baca, Ph.D.

Enunciado: `Lab12b_SRGAN_Programacion.pdf`. En este laboratorio **ustedes escriben** cada componente de SRGAN. Cada tarea termina con una celda `verificar_...()` que ejecuta pruebas automáticas: **no la modifiquen**. Si la prueba pasa, verán ✅.

| Tarea | Qué implementan | Puntos |
|---|---|---|
| 1 | `pixel_shuffle_manual` (sin usar `nn.PixelShuffle`) | 1.5 |
| 2 | Inicialización `icnr_` | 1.5 |
| 3 | `BloqueResidual`, `BloqueSubpixel`, `SRResNet` | 3 |
| 4 | `Discriminador` | 2 |
| 5 | `PerdidaVGG` | 2 |
| 6 | `paso_srresnet` y entrenamiento con MSE | 2 |
| 7 | `paso_srgan` y entrenamiento adversarial | 4 |
| 8 | `interpolar_redes` | 1.5 |
| 9 | `error_temporal` y video lado a lado | 1.5 |
| — | Notebook que corre de principio a fin, código limpio | 1 |

**Use GPU** (T4). Los dos entrenamientos (Tareas 6 y 7) toman unos minutos cada uno: mientras corren, avancen con las tareas siguientes.

## Preparación (dada, no modificar)

In [ ]:
import os, math, time, copy, random, base64
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import matplotlib.pyplot as plt
from skimage import data as skdata
from skimage.metrics import structural_similarity
from IPython.display import HTML, display

dispositivo = 'cuda' if torch.cuda.is_available() else 'cpu'
torch.manual_seed(0); np.random.seed(0); random.seed(0)
torch.backends.cudnn.benchmark = True
print('PyTorch', torch.__version__, '| dispositivo:', dispositivo)

# Solo para verificar el notebook en CPU con muy pocas iteraciones (dejar en '0' en clase)
MODO_PRUEBA = os.environ.get('SRGAN_MODO_PRUEBA', '0') == '1'
ESCALA = 4

# ----------------------------------------------------------------- utilidades
def a_tensor(img):
    # uint8 (H, W, 3) -> float (1, 3, H, W) en [0, 1]
    return torch.from_numpy(np.ascontiguousarray(img)).permute(2, 0, 1)[None].float() / 255

def a_imagen(t):
    # float (1, 3, H, W) o (3, H, W) en [0, 1] -> uint8 (H, W, 3)
    t = t.detach().float().cpu().clamp(0, 1)
    if t.dim() == 4: t = t[0]
    return (t.permute(1, 2, 0).numpy() * 255 + 0.5).astype(np.uint8)

def recortar_multiplo(img, s=ESCALA):
    h, w = img.shape[:2]
    return img[:h - h % s, :w - w % s]

def reducir(hr01, s=ESCALA):
    # Degradación del artículo: submuestreo bicúbico con antialiasing (como Pillow / MATLAB imresize)
    return F.interpolate(hr01, scale_factor=1 / s, mode='bicubic', antialias=True, align_corners=False).clamp(0, 1)

def ampliar(lr01, s=ESCALA, modo='bicubic'):
    if modo == 'nearest':
        return F.interpolate(lr01, scale_factor=s, mode='nearest')
    return F.interpolate(lr01, scale_factor=s, mode=modo, align_corners=False).clamp(0, 1)

def canal_y(img):
    # Luminancia Y de YCbCr (ITU-R BT.601), en [16, 235]: así se reportan PSNR y SSIM en superresolución
    img = img.astype(np.float64) / 255
    return 16 + 65.481 * img[..., 0] + 128.553 * img[..., 1] + 24.966 * img[..., 2]

def psnr_y(a, b, borde=ESCALA):
    ya, yb = canal_y(a)[borde:-borde, borde:-borde], canal_y(b)[borde:-borde, borde:-borde]
    mse = np.mean((ya - yb) ** 2)
    return float('inf') if mse == 0 else 10 * np.log10(255 ** 2 / mse)

def ssim_y(a, b, borde=ESCALA):
    ya, yb = canal_y(a)[borde:-borde, borde:-borde], canal_y(b)[borde:-borde, borde:-borde]
    return structural_similarity(ya, yb, data_range=255, gaussian_weights=True, sigma=1.5, use_sample_covariance=False)

def mostrar(imgs, titulos, ancho=4, fila_alto=None):
    fig, axes = plt.subplots(1, len(imgs), figsize=(ancho * len(imgs), fila_alto or ancho))
    for ax, im, t in zip(np.atleast_1d(axes), imgs, titulos):
        ax.imshow(im, interpolation='nearest'); ax.set_title(t, fontsize=10); ax.axis('off')
    plt.tight_layout(); plt.show()

# ----------------------------------------------------------------- imágenes de prueba (vienen con scikit-image)
PRUEBA = {nombre: recortar_multiplo(f()) for nombre, f in [
    ('astronauta', skdata.astronaut), ('café', skdata.coffee), ('gato', skdata.chelsea),
    ('cohete', skdata.rocket), ('histología', skdata.immunohistochemistry)]}
for k, v in PRUEBA.items(): print(f'{k:12s} {v.shape}')

In [ ]:
from torchvision.datasets import Flowers102

N_IMAGENES, RECORTES_POR_IMAGEN, TAM_HR = (40, 4, 96) if MODO_PRUEBA else (3000, 4, 96)

def banco_de_parches(imagenes, por_imagen=RECORTES_POR_IMAGEN, tam=TAM_HR):
    parches = []
    for im in imagenes:
        im = np.asarray(im.convert('RGB')) if hasattr(im, 'convert') else im
        h, w = im.shape[:2]
        if min(h, w) < tam: continue
        for _ in range(por_imagen):
            y, x = np.random.randint(0, h - tam + 1), np.random.randint(0, w - tam + 1)
            parches.append(im[y:y + tam, x:x + tam])
    return torch.from_numpy(np.stack(parches)).permute(0, 3, 1, 2).contiguous()   # uint8 (N, 3, 96, 96)

t0 = time.time()
if MODO_PRUEBA:
    fuente = [skdata.hubble_deep_field(), skdata.retina()[::2, ::2], skdata.rocket()] * (N_IMAGENES // 3)
else:
    divisiones = [Flowers102('datos', split=s, download=True) for s in ('train', 'val', 'test')]
    todas = [(d, i) for d in divisiones for i in range(len(d))]
    random.shuffle(todas)
    fuente = (d[i][0] for d, i in todas[:N_IMAGENES])
BANCO = banco_de_parches(fuente)
print(f'Banco: {tuple(BANCO.shape)}  ({BANCO.numel() / 1e6:.0f} MB, uint8)  en {time.time() - t0:.0f}s')

def lote(n=16):
    # Devuelve (lr en [0,1], hr en [-1,1]) en la GPU, con volteos y rotaciones aleatorias
    idx = torch.randint(0, len(BANCO), (n,))
    hr01 = BANCO[idx].to(dispositivo, non_blocking=True).float() / 255
    if random.random() < 0.5: hr01 = hr01.flip(3)
    hr01 = torch.rot90(hr01, random.randint(0, 3), dims=(2, 3))
    return reducir(hr01), hr01 * 2 - 1

lr_b, hr_b = lote(8)
print('LR:', tuple(lr_b.shape), f'[{lr_b.min():.2f}, {lr_b.max():.2f}]   HR:', tuple(hr_b.shape), f'[{hr_b.min():.2f}, {hr_b.max():.2f}]')
mostrar([a_imagen((hr_b[i] + 1) / 2) for i in range(6)], ['HR 96×96'] * 6, ancho=2.2)
mostrar([a_imagen(lr_b[i]) for i in range(6)], ['LR 24×24'] * 6, ancho=2.2)

In [ ]:
import inspect
bce = nn.BCEWithLogitsLoss()
VAL = a_tensor(PRUEBA['astronauta'][:256, :256]); VAL_LR = reducir(VAL); VAL_HR = a_imagen(VAL)
PSNR_BICUBICA_VAL = psnr_y(a_imagen(ampliar(VAL_LR)), VAL_HR)

@torch.no_grad()
def superresolver(G, lr01):
    # lr01 (N, 3, h, w) en [0,1] -> (N, 3, 4h, 4w) en [0,1], con BN en modo evaluación
    G.eval()
    return ((G(lr01.to(dispositivo)) + 1) / 2).clamp(0, 1).cpu()

def contar(m): return sum(p.numel() for p in m.parameters())
def ok(msg): print('✅', msg)
print(f'PSNR bicúbica en validación: {PSNR_BICUBICA_VAL:.2f} dB')

## Tarea 1 — *Pixel shuffle* a mano (1.5 pts)
Implemente $\mathcal{PS}$ para un factor $r$: un tensor $(N,\,C r^2,\,H,\,W)$ se convierte en $(N,\,C,\,rH,\,rW)$ con
$$\mathcal{PS}(T)_{n,c,y,x}=T_{n,\;c r^2+r(y\bmod r)+(x\bmod r),\;\lfloor y/r\rfloor,\;\lfloor x/r\rfloor}.$$
Use solo `view`/`reshape` y `permute`. **Prohibido** usar `nn.PixelShuffle` o `F.pixel_shuffle`.

In [ ]:
def pixel_shuffle_manual(t, r):
    N, C, H, W = t.shape
    # TODO: 1) separe los canales en (c, r, r)   2) reordene los ejes   3) junte (H, r) y (W, r)
    raise NotImplementedError

In [ ]:
def verificar_1():
    fuente = inspect.getsource(pixel_shuffle_manual)
    assert 'PixelShuffle' not in fuente and 'pixel_shuffle(' not in fuente.replace('def pixel_shuffle_manual(', ''), 'No use nn.PixelShuffle ni F.pixel_shuffle'
    for r, C in [(2, 4), (2, 12), (3, 18), (4, 48)]:
        t = torch.randn(2, C, 5, 7)
        s = pixel_shuffle_manual(t, r)
        assert s.shape == (2, C // r**2, 5 * r, 7 * r), f'forma incorrecta para r={r}: {tuple(s.shape)}'
        assert torch.equal(s, F.pixel_shuffle(t, r)), f'valores incorrectos para r={r}'
    ok('Tarea 1: pixel_shuffle_manual es correcto')
verificar_1()

## Tarea 2 — Inicialización ICNR (1.5 pts)
`conv` es una `nn.Conv2d` con `c·r²` canales de salida que irá seguida de un *pixel shuffle*. Inicialice `c` filtros con `nn.init.kaiming_normal_` y **cópielos** para que los `r²` canales consecutivos de cada grupo sean idénticos. Haga lo mismo con el sesgo (si existe). Resultado esperado: al inicio, `PS(conv(x))` = "convolución con `c` filtros + ampliación por vecino más cercano".

In [ ]:
def icnr_(conv, r=2):
    # TODO (pista: repeat_interleave; recuerde usar torch.no_grad() y copy_)
    raise NotImplementedError

In [ ]:
def verificar_2():
    torch.manual_seed(0)
    for r, c in [(2, 3), (2, 64), (3, 5)]:
        conv = nn.Conv2d(16, c * r * r, 3, 1, 1); icnr_(conv, r)
        x = torch.randn(2, 16, 9, 9)
        with torch.no_grad():
            izq = F.pixel_shuffle(conv(x), r)
            der = F.interpolate(F.conv2d(x, conv.weight[::r * r], conv.bias[::r * r], padding=1), scale_factor=r, mode='nearest')
        assert torch.allclose(izq, der, atol=1e-5), f'PS(conv(x)) no es igual a vecino(conv(x)) con r={r}'
        assert conv.weight.std() > 0.01, 'los pesos parecen no inicializados'
    ok('Tarea 2: icnr_ es correcto')
verificar_2()

## Tarea 3 — Generador SRResNet (3 pts)
Implemente exactamente la arquitectura de Ledig et al. (Figura 6 de las notas):
* **`BloqueResidual(c)`**: conv 3×3 → BN → PReLU → conv 3×3 → BN, más la entrada (salto).
* **`BloqueSubpixel(c, r=2)`**: conv 3×3 de `c` a `c·r²` canales (inicializada con `icnr_`) → *pixel shuffle* (aquí sí puede usar `nn.PixelShuffle`) → PReLU.
* **`SRResNet(B=16, c=64, escala=4)`**: conv 9×9 (3→c) + PReLU → `B` bloques residuales → conv 3×3 + BN → **suma con la salida de la primera capa** (salto global) → `log2(escala)` bloques sub-píxel → conv 9×9 (c→3) → `tanh`.

Use `nn.PReLU()` (un solo parámetro) y `padding` para conservar el tamaño.

In [ ]:
class BloqueResidual(nn.Module):
    def __init__(self, c=64):
        super().__init__()
        # TODO
        pass
    def forward(self, x):
        raise NotImplementedError

class BloqueSubpixel(nn.Module):
    def __init__(self, c=64, r=2):
        super().__init__()
        # TODO
        pass
    def forward(self, x):
        raise NotImplementedError

class SRResNet(nn.Module):
    def __init__(self, B=16, c=64, escala=4):
        super().__init__()
        # TODO
        pass
    def forward(self, lr01):
        # TODO: no olvide el salto global ni la tanh final
        raise NotImplementedError

In [ ]:
def verificar_3():
    torch.manual_seed(0)
    b = BloqueResidual(8); x = torch.randn(2, 8, 6, 6)
    assert b(x).shape == x.shape, 'BloqueResidual debe conservar la forma'
    assert sum(isinstance(m, nn.BatchNorm2d) for m in b.modules()) == 2, 'BloqueResidual lleva 2 BatchNorm'
    s = BloqueSubpixel(8)
    assert s(x).shape == (2, 8, 12, 12), 'BloqueSubpixel debe duplicar la resolución y conservar los canales'
    G = SRResNet()
    assert contar(G) == 1_549_462, f'SRResNet(B=16, c=64) debe tener 1,549,462 parámetros; tiene {contar(G):,}'
    y = G(torch.rand(2, 3, 24, 24))
    assert y.shape == (2, 3, 96, 96), f'salida {tuple(y.shape)}; se esperaba (2, 3, 96, 96)'
    assert y.abs().max() <= 1, 'la salida debe estar en [-1, 1] (tanh)'
    assert SRResNet(B=2, escala=2)(torch.rand(1, 3, 10, 14)).shape == (1, 3, 20, 28), 'escala=2 debe usar un solo bloque sub-píxel'
    assert SRResNet(B=2, escala=8)(torch.rand(1, 3, 5, 5)).shape == (1, 3, 40, 40), 'escala=8 debe usar tres bloques sub-píxel'
    # salto global: si anulamos los bloques residuales y la capa media, la red debe seguir dependiendo de la entrada
    with torch.no_grad():
        for p in G.medio.parameters(): p.zero_()
    G.eval(); a, b2 = G(torch.zeros(1, 3, 8, 8)), G(torch.ones(1, 3, 8, 8))
    assert not torch.allclose(a, b2), 'falta el salto global (la salida no depende de la entrada cuando la capa media es cero)'
    ok('Tarea 3: SRResNet es correcta')
verificar_3()

## Tarea 4 — Discriminador (2 pts)
Ocho convoluciones 3×3 con LeakyReLU(0.2): canales y pasos
`(3→64, s1), (64→64, s2), (64→128, s1), (128→128, s2), (128→256, s1), (256→256, s2), (256→512, s1), (512→512, s2)`.
BatchNorm en todas **excepto la primera**. Luego `nn.AdaptiveAvgPool2d(6)`, aplanar, densa 512·6·6→1024, LeakyReLU(0.2), densa 1024→1. La salida es un **logit** (sin sigmoide: usaremos `BCEWithLogitsLoss`).

In [ ]:
class Discriminador(nn.Module):
    def __init__(self):
        super().__init__()
        # TODO
        pass
    def forward(self, x11):
        raise NotImplementedError

In [ ]:
def verificar_4():
    D = Discriminador()
    assert contar(D) == 23_565_505, f'el discriminador debe tener 23,565,505 parámetros; tiene {contar(D):,}'
    convs = [m for m in D.modules() if isinstance(m, nn.Conv2d)]
    assert len(convs) == 8 and [c.stride[0] for c in convs] == [1, 2] * 4, 'deben ser 8 convoluciones con pasos 1,2,1,2,...'
    assert sum(isinstance(m, nn.BatchNorm2d) for m in D.modules()) == 7, 'BN en todas las convoluciones menos la primera'
    assert not any(isinstance(m, (nn.Sigmoid, nn.MaxPool2d)) for m in D.modules()), 'sin sigmoide (logit) y sin max-pooling'
    assert D(torch.randn(3, 3, 96, 96)).shape == (3, 1) and D(torch.randn(2, 3, 128, 128)).shape == (2, 1), 'la salida debe ser (N, 1)'
    ok('Tarea 4: Discriminador es correcto')
verificar_4()

## Tarea 5 — Pérdida VGG (2 pts)
`PerdidaVGG(capa)` con `capa` en `{'22', '54'}`:
* tome `vgg19(weights=VGG19_Weights.IMAGENET1K_V1).features` **truncada justo después** de la ReLU de `conv2_2` (VGG22) o de `conv5_4` (VGG54). Encuentre usted los índices imprimiendo `vgg19().features`;
* congele los pesos (`requires_grad=False`) y déjela en modo `eval`;
* `forward(sr, hr)`: ambas imágenes vienen en $[-1,1]$; páselas a $[0,1]$, normalícelas con la media `[0.485, 0.456, 0.406]` y la desviación `[0.229, 0.224, 0.225]` de ImageNet, y devuelva la MSE entre las características. `hr` no necesita gradiente.

In [ ]:
from torchvision.models import vgg19, VGG19_Weights
print(vgg19().features)

class PerdidaVGG(nn.Module):
    def __init__(self, capa='54'):
        super().__init__()
        # TODO
        pass
    def forward(self, sr, hr):
        raise NotImplementedError
    def train(self, modo=True):
        return super().train(False)          # la VGG siempre en modo evaluación

In [ ]:
def verificar_5():
    v22, v54 = PerdidaVGG('22'), PerdidaVGG('54')
    assert len(v22.f) == 9 and len(v54.f) == 36, f'cortes incorrectos: {len(v22.f)} y {len(v54.f)} capas (VGG22 termina en relu2_2, VGG54 en relu5_4)'
    assert isinstance(v22.f[-1], nn.ReLU) and isinstance(v54.f[-1], nn.ReLU), 'la última capa debe ser la ReLU'
    assert all(not p.requires_grad for p in v54.parameters()), 'la VGG debe estar congelada'
    x = torch.rand(1, 3, 64, 64) * 2 - 1
    assert v54(x, x).item() == 0, 'la pérdida de una imagen consigo misma debe ser 0'
    sr = (x + 0.1 * torch.randn_like(x)).requires_grad_(); hr = x.clone().requires_grad_()
    v54(sr, hr).backward()
    assert sr.grad is not None and sr.grad.abs().sum() > 0, 'el gradiente debe llegar a sr'
    assert hr.grad is None, 'hr no debe recibir gradiente (use .detach())'
    ref = F.mse_loss(v22.f(((sr + 1) / 2 - v22.media) / v22.desv), v22.f(((x + 1) / 2 - v22.media) / v22.desv)) if hasattr(v22, 'media') else None
    if ref is not None: assert torch.allclose(v22(sr, x), ref), 'revise la normalización'
    v54.train(); assert not v54.f.training, 'la VGG debe quedar en modo eval aunque se llame .train()'
    ok('Tarea 5: PerdidaVGG es correcta')
verificar_5()
vgg54 = PerdidaVGG('54').to(dispositivo)

## Tarea 6 — Entrenar SRResNet con MSE (2 pts)
Implemente **un paso** de entrenamiento: modo entrenamiento, *forward*, MSE en píxeles (rango $[-1,1]$), `zero_grad`, `backward`, `step`. Devuelva la pérdida como `float`. El bucle de entrenamiento está dado.

In [ ]:
def paso_srresnet(G, opt, lr, hr):
    # TODO
    raise NotImplementedError

In [ ]:
def verificar_6():
    torch.manual_seed(0)
    G = SRResNet(B=2, c=16); opt = torch.optim.Adam(G.parameters(), 1e-3)
    lr, hr = torch.rand(4, 3, 12, 12), torch.rand(4, 3, 48, 48) * 2 - 1
    antes = [p.clone() for p in G.parameters()]
    perdidas = [paso_srresnet(G, opt, lr, hr) for _ in range(30)]
    assert isinstance(perdidas[0], float), 'devuelva loss.item()'
    assert any(not torch.equal(a, p) for a, p in zip(antes, G.parameters())), 'los parámetros no cambiaron'
    assert perdidas[-1] < perdidas[0], 'en un lote fijo la pérdida debería bajar'
    ok('Tarea 6: paso_srresnet es correcto')
verificar_6()

In [ ]:
ITER_SRRESNET = 30 if MODO_PRUEBA else 5000
torch.manual_seed(0)
srresnet = SRResNet().to(dispositivo)
opt = torch.optim.Adam(srresnet.parameters(), lr=1e-4)
psnr_inicial = psnr_y(a_imagen(superresolver(srresnet, VAL_LR)), VAL_HR)
t0 = time.time()
for paso in range(1, ITER_SRRESNET + 1):
    loss = paso_srresnet(srresnet, opt, *lote(16))
    if paso % (10 if MODO_PRUEBA else 1000) == 0:
        p = psnr_y(a_imagen(superresolver(srresnet, VAL_LR)), VAL_HR)
        print(f'paso {paso:5d} | MSE {loss:.4f} | PSNR val {p:.2f} dB (bicúbica {PSNR_BICUBICA_VAL:.2f}) | {time.time() - t0:.0f}s')
psnr_srresnet = psnr_y(a_imagen(superresolver(srresnet, VAL_LR)), VAL_HR)
assert psnr_srresnet > psnr_inicial + (0 if MODO_PRUEBA else 5), 'la PSNR de validación casi no mejoró: revise la Tarea 6'
ok(f'SRResNet entrenada: PSNR val {psnr_srresnet:.2f} dB')

## Tarea 7 — Un paso de SRGAN (4 pts)
Implemente **un paso** del entrenamiento alternado ($k=1$), en este orden:
1. `G.train(); D.train()`; genere `sr = G(lr)`.
2. **Paso de D:** `bce(D(hr), 1) + bce(D(sr.detach()), 0)`; `zero_grad`, `backward`, `step` de `optD`. ¿Por qué `detach`?
3. **Paso de G:** `contenido = perdida_contenido(sr, hr)`, `adversarial = bce(D(sr), 1)` (pérdida **no saturante**), `total = contenido + peso_adv * adversarial`; `zero_grad`, `backward`, `step` de `optG`.
4. Devuelva un diccionario con las claves `'D'`, `'contenido'`, `'adversarial'`, `'D(x)'`, `'D(G(z))'` (estas dos: promedio de la sigmoide de los logits del paso 2), todas como `float`.

Use `bce = nn.BCEWithLogitsLoss()` (definida en la preparación) y `torch.ones_like` / `torch.zeros_like` para las etiquetas.

In [ ]:
def paso_srgan(G, D, optG, optD, lr, hr, perdida_contenido, peso_adv=1e-3):
    # TODO
    raise NotImplementedError

In [ ]:
def verificar_7():
    torch.manual_seed(0)
    G, D = SRResNet(B=1, c=8), Discriminador()
    lr, hr = torch.rand(4, 3, 24, 24), torch.rand(4, 3, 96, 96) * 2 - 1
    mse = lambda a, b: F.mse_loss(a, b)
    # (a) claves y tipos
    optG, optD = torch.optim.Adam(G.parameters(), 1e-4), torch.optim.Adam(D.parameters(), 1e-4)
    out = paso_srgan(G, D, optG, optD, lr, hr, mse)
    assert set(out) == {'D', 'contenido', 'adversarial', 'D(x)', 'D(G(z))'}, f'claves: {set(out)}'
    assert all(isinstance(v, float) and math.isfinite(v) for v in out.values()), 'todos los valores deben ser float finitos'
    assert 0 < out['D(x)'] < 1 and 0 < out['D(G(z))'] < 1, 'D(x) y D(G(z)) son probabilidades'
    # (b) ambos jugadores se actualizan
    gA, dA = [p.clone() for p in G.parameters()], [p.clone() for p in D.parameters()]
    paso_srgan(G, D, optG, optD, lr, hr, mse)
    assert any(not torch.equal(a, p) for a, p in zip(gA, G.parameters())), 'G no se actualizó'
    assert any(not torch.equal(a, p) for a, p in zip(dA, D.parameters())), 'D no se actualizó'
    # (c) con peso_adv = 0 y D congelado, G solo minimiza el contenido: debe coincidir con un paso de MSE puro
    torch.manual_seed(1); G1 = SRResNet(B=1, c=8); G2 = copy.deepcopy(G1)
    o1 = torch.optim.SGD(G1.parameters(), 0.1); o2 = torch.optim.SGD(G2.parameters(), 0.1)
    paso_srgan(G1, Discriminador(), o1, torch.optim.SGD(D.parameters(), 0.0), lr, hr, mse, peso_adv=0.0)
    G2.train(); l = mse(G2(lr), hr); o2.zero_grad(); l.backward(); o2.step()
    assert all(torch.allclose(a, b, atol=1e-6) for a, b in zip(G1.parameters(), G2.parameters())), \
        'con peso_adv=0 el paso de G debe ser idéntico a un paso de MSE puro: ¿hizo optG.zero_grad() antes de backward? ¿combinó bien las pérdidas?'
    ok('Tarea 7: paso_srgan es correcto')
verificar_7()

In [ ]:
ITER_SRGAN = 20 if MODO_PRUEBA else 3000
torch.manual_seed(0)
srgan, D = copy.deepcopy(srresnet), Discriminador().to(dispositivo)        # G se inicializa con la SRResNet
optG, optD = torch.optim.Adam(srgan.parameters(), 1e-4), torch.optim.Adam(D.parameters(), 1e-4)
hist, t0 = [], time.time()
for paso in range(1, ITER_SRGAN + 1):
    if paso == ITER_SRGAN // 2 + 1:                                         # 1e-4 -> 1e-5 a la mitad
        for o in (optG, optD):
            for g in o.param_groups: g['lr'] = 1e-5
    hist.append(paso_srgan(srgan, D, optG, optD, *lote(16), perdida_contenido=vgg54))
    if paso % (10 if MODO_PRUEBA else 500) == 0:
        m = {k: np.mean([h[k] for h in hist[-500:]]) for k in hist[0]}
        print(f'paso {paso:5d} | ' + ' | '.join(f'{k} {v:.3f}' for k, v in m.items()) + f' | {time.time() - t0:.0f}s')

fig, axes = plt.subplots(1, 3, figsize=(15, 3))
suav = lambda k: np.convolve([h[k] for h in hist], np.ones(50) / 50, mode='valid') if len(hist) > 50 else [h[k] for h in hist]
axes[0].plot(suav('contenido')); axes[0].set_title('contenido (VGG54)')
axes[1].plot(suav('adversarial'), label='G adversarial'); axes[1].plot(suav('D'), label='D'); axes[1].legend()
axes[2].plot(suav('D(x)'), label='D(x)'); axes[2].plot(suav('D(G(z))'), label='D(G(z))'); axes[2].legend()
for ax in axes: ax.set_xlabel('paso'); ax.grid(alpha=.3)
plt.tight_layout(); plt.show()

r0, c0, t = 60, 100, 112
rec = lambda im: im[r0:r0 + t, c0:c0 + t]
mostrar([rec(VAL_HR), rec(a_imagen(ampliar(VAL_LR))), rec(a_imagen(superresolver(srresnet, VAL_LR))), rec(a_imagen(superresolver(srgan, VAL_LR)))],
        ['HR', 'bicúbica', 'SRResNet', 'SRGAN'], ancho=3.5)

## Tarea 8 — Interpolación de redes (1.5 pts)
Devuelva una **copia** de `G_a` cuyos tensores del `state_dict` sean $(1-\alpha)\,\theta_a+\alpha\,\theta_b$. Los tensores enteros (como `num_batches_tracked` de la BatchNorm) se copian de `G_a` sin interpolar. `G_a` y `G_b` no deben modificarse.

In [ ]:
def interpolar_redes(G_a, G_b, alfa):
    # TODO
    raise NotImplementedError

In [ ]:
def verificar_8():
    torch.manual_seed(0)
    A, B = SRResNet(B=1, c=8), SRResNet(B=1, c=8)
    A.train(); A(torch.rand(2, 3, 6, 6))                     # cambia running_mean y num_batches_tracked de A
    sa0 = {k: v.clone() for k, v in A.state_dict().items()}
    for alfa in [0.0, 0.3, 1.0]:
        sm, sa, sb = interpolar_redes(A, B, alfa).state_dict(), A.state_dict(), B.state_dict()
        for k in sa:
            if sa[k].is_floating_point():
                assert torch.allclose(sm[k], (1 - alfa) * sa[k] + alfa * sb[k], atol=1e-6), f'{k} mal interpolado con alfa={alfa}'
            else:
                assert torch.equal(sm[k], sa[k]), f'{k} es entero: debe copiarse de G_a'
    assert all(torch.equal(sa0[k], v) for k, v in A.state_dict().items()), 'G_a fue modificada'
    ok('Tarea 8: interpolar_redes es correcto')
verificar_8()

ALFAS = [0, 0.25, 0.5, 0.75, 1.0]
puntos = []
for alfa in ALFAS:
    G_alfa = interpolar_redes(srresnet, srgan, alfa)
    sr = a_imagen(superresolver(G_alfa, VAL_LR))
    puntos.append((psnr_y(sr, VAL_HR), ssim_y(sr, VAL_HR)))
    print(f'alfa={alfa:.2f}  PSNR {puntos[-1][0]:.2f} dB  SSIM {puntos[-1][1]:.3f}')

## Tarea 9 — Error temporal y video lado a lado (1.5 pts)
`cuadros` y `ref` son listas de imágenes `uint8` de igual forma (un video superresuelto y el HR). Implemente
$$E_{\text{temp}}=\frac{1}{T-1}\sum_{t=2}^{T}\operatorname{media}\Big|\big(\hat I_t-\hat I_{t-1}\big)-\big(I_t-I_{t-1}\big)\Big|$$
(media sobre todos los píxeles y canales; **cuidado** con el desbordamiento de `uint8`: convierta a `float32`).

Luego ejecute la celda que genera el video lado a lado (bicúbica | SRResNet | SRGAN | HR) e imprime $E_{\text{temp}}$ de cada método.

In [ ]:
def error_temporal(cuadros, ref):
    # TODO
    raise NotImplementedError

In [ ]:
def verificar_9():
    rng = np.random.default_rng(0)
    ref = [rng.integers(0, 256, (8, 8, 3), dtype=np.uint8) for _ in range(5)]
    assert error_temporal(ref, ref) == 0, 'un video idéntico al de referencia tiene error 0'
    ref2 = [np.clip(r.astype(int), 0, 248).astype(np.uint8) for r in ref]
    desp2 = [r + 7 for r in ref2]
    assert abs(error_temporal(desp2, ref2)) < 1e-6, 'un brillo constante en todos los cuadros no es parpadeo: el error debe ser 0'
    parpadeo = [r2 + (7 if t % 2 else 0) for t, r2 in enumerate(ref2)]
    assert abs(error_temporal(parpadeo, ref2) - 7.0) < 1e-5, 'un parpadeo de ±7 niveles debe dar E_temp = 7'
    ok('Tarea 9: error_temporal es correcto')
verificar_9()

In [ ]:
import cv2, imageio

def video_panoramico(img, alto=192, ancho=256, n=48):
    H, W = img.shape[:2]
    cuadros = []
    for t in range(n):
        s = 0.5 - 0.5 * np.cos(np.pi * t / (n - 1))                  # 0 -> 1 con arranque y frenado suaves
        dx, dy = s * (W - ancho - 1), 0.35 * (H - alto - 1) * np.sin(np.pi * t / (n - 1))
        M = np.float32([[1, 0, -dx], [0, 1, -dy]])                   # traslación sub-píxel
        cuadros.append(cv2.warpAffine(img, M, (ancho, alto), flags=cv2.INTER_CUBIC, borderMode=cv2.BORDER_REFLECT))
    return np.stack(cuadros)

def superresolver_video(cuadros_lr01, G, lote_cuadros=8):
    # cuadros_lr01: tensor (T, 3, h, w) en [0,1]  ->  lista de uint8 (4h, 4w, 3)
    salida = []
    for i in range(0, len(cuadros_lr01), lote_cuadros):
        sr = superresolver(G, cuadros_lr01[i:i + lote_cuadros])
        salida += [a_imagen(s) for s in sr]
    return salida

def rotular(img, texto):
    img = img.copy()
    cv2.putText(img, texto, (6, 18), cv2.FONT_HERSHEY_SIMPLEX, 0.5, (0, 0, 0), 3, cv2.LINE_AA)
    cv2.putText(img, texto, (6, 18), cv2.FONT_HERSHEY_SIMPLEX, 0.5, (255, 255, 255), 1, cv2.LINE_AA)
    return img

def guardar_y_mostrar(cuadros, ruta, fps=12, ancho_px=1100):
    imageio.v2.mimwrite(ruta, cuadros, format='FFMPEG', fps=fps, codec='libx264', quality=8, macro_block_size=1)
    b64 = base64.b64encode(open(ruta, 'rb').read()).decode()
    display(HTML(f'<video width="{ancho_px}" controls loop autoplay muted><source src="data:video/mp4;base64,{b64}" type="video/mp4"></video>'))

HR_V = video_panoramico(PRUEBA['café'], n=16 if MODO_PRUEBA else 48)
LR_V = reducir(torch.from_numpy(HR_V).permute(0, 3, 1, 2).float() / 255)
VIDEOS = {'bicúbica': [a_imagen(c) for c in ampliar(LR_V)],
          'SRResNet': superresolver_video(LR_V, srresnet),
          'SRGAN': superresolver_video(LR_V, srgan),
          'HR': list(HR_V)}
for k in ['bicúbica', 'SRResNet', 'SRGAN']:
    print(f'{k:10s} E_temp = {error_temporal(VIDEOS[k], VIDEOS["HR"]):6.2f}')
guardar_y_mostrar([np.concatenate([rotular(VIDEOS[k][t], k) for k in VIDEOS], axis=1) for t in range(len(HR_V))], 'lab12b_video.mp4')

## Resumen de verificación
Ejecute esta celda al final: todas las pruebas deben pasar en un notebook que corre de principio a fin (**Entorno de ejecución → Reiniciar y ejecutar todo**).

In [ ]:
for i, v in enumerate([verificar_1, verificar_2, verificar_3, verificar_4, verificar_5, verificar_6, verificar_7, verificar_8, verificar_9], 1):
    try: v()
    except Exception as e: print(f'❌ Tarea {i}: {type(e).__name__}: {e}')